In [ ]:
# ==========================================
# EMAN'S SFT PIPELINE (Supervised Fine-Tuning)
# Complete Code - All 18 Code Cells Combined
# ==========================================

# --- Cell 1: Explore available files ---
import os
print("Input directory files:", os.listdir('/kaggle/input') if os.path.exists('/kaggle/input') else "No /kaggle/input")
print("Working directory files:", os.listdir('/kaggle/working'))

# --- Cell 2: Read and inspect SFT data ---
import pandas as pd
sft_df = pd.read_csv('SFT_Pairs_300.csv')
print("SFT Dataset Shape:", sft_df.shape)
print("Columns:", sft_df.columns.tolist())
display(sft_df.head())

# --- Cell 3: Verify splits and missing values ---
print("Split counts:\n", sft_df['Split'].value_counts())
print("Missing values:\n", sft_df.isnull().sum())

# --- Cell 4: Split into train and validation DataFrames ---
train_df = sft_df[sft_df['Split'] == 'train'].reset_index(drop=True)
val_df = sft_df[sft_df['Split'] == 'val'].reset_index(drop=True)
print(f"Train samples: {len(train_df)}, Val samples: {len(val_df)}")

# --- Cell 5: Verify cpt_best checkpoint path ---
print("Checking cpt_best path:", os.path.exists('/kaggle/working/cpt_best'))

# --- Cell 6: Inspect working directory contents ---
print(os.listdir('/kaggle/working'))

# --- Cell 7: Additional checkpoint path verification ---
print("Checkpoint directory contents:", os.listdir('/kaggle/working/cpt_best') if os.path.exists('/kaggle/working/cpt_best') else "Not found")

# --- Cell 8: Load model and tokenizer ---
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model_path = '/kaggle/working/cpt_best' if os.path.exists('/kaggle/working/cpt_best') else 'gpt2'
tokenizer = GPT2Tokenizer.from_pretrained(model_path)
model = GPT2LMHeadModel.from_pretrained(model_path)

# --- Cell 9: Confirmation check ---
print("Model and tokenizer loaded successfully:", model is not None)

# --- Cell 10: Add special tokens ---
special_tokens = {'additional_special_tokens': ['<|user|>', '<|assistant|>', '<|end|>']}
tokenizer.add_special_tokens(special_tokens)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model.resize_token_embeddings(len(tokenizer))
model.to(device)
print("New vocab size:", len(tokenizer))

# --- Cell 11: Prompt-Label Masking function ---
class SFTDataset(torch.utils.data.Dataset):
    def __init__(self, df, tokenizer, max_length=512):
        self.examples = []
        for _, row in df.iterrows():
            prompt = f"<|user|>{row['Question']}<|assistant|>"
            answer = f"{row['Answer']}<|end|>"
            
            prompt_ids = tokenizer.encode(prompt, add_special_tokens=False)
            answer_ids = tokenizer.encode(answer, add_special_tokens=False)
            input_ids = prompt_ids + answer_ids
            
            # Mask prompt tokens with -100 so loss is computed only on answer tokens
            labels = [-100] * len(prompt_ids) + answer_ids
            
            if len(input_ids) > max_length:
                input_ids = input_ids[:max_length]
                labels = labels[:max_length]
                
            self.examples.append({
                'input_ids': input_ids,
                'labels': labels
            })
            
    def __len__(self):
        return len(self.examples)
    
    def __getitem__(self, idx):
        return self.examples[idx]

print("SFTDataset class defined.")

# --- Cell 12: DataLoader & Collate function ---
def collate_sft(batch, pad_token_id):
    max_len = max(len(item['input_ids']) for item in batch)
    input_ids, labels = [], []
    for item in batch:
        pad_len = max_len - len(item['input_ids'])
        input_ids.append(item['input_ids'] + [pad_token_id] * pad_len)
        labels.append(item['labels'] + [-100] * pad_len)
    return {
        'input_ids': torch.tensor(input_ids, dtype=torch.long),
        'labels': torch.tensor(labels, dtype=torch.long)
    }

train_dataset = SFTDataset(train_df, tokenizer)
val_dataset = SFTDataset(val_df, tokenizer)

train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=4, shuffle=True, 
    collate_fn=lambda b: collate_sft(b, tokenizer.pad_token_id)
)
val_loader = torch.utils.data.DataLoader(
    val_dataset, batch_size=4, shuffle=False, 
    collate_fn=lambda b: collate_sft(b, tokenizer.pad_token_id)
)
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

# --- Cell 13: Setup optimizer ---
from transformers import AdamW
optimizer = AdamW(model.parameters(), lr=5e-5)
torch.manual_seed(42)
print("Optimizer AdamW initialized with lr=5e-5 and seed=42.")

# --- Cell 14: Training Loop (3 Epochs) ---
for epoch in range(1, 4):
    model.train()
    total_train_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        labels = batch['labels'].to(device)
        outputs = model(input_ids=input_ids, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        total_train_loss += loss.item()
        
    avg_train_loss = total_train_loss / len(train_loader)
    
    model.eval()
    total_val_loss = 0
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            labels = batch['labels'].to(device)
            outputs = model(input_ids=input_ids, labels=labels)
            total_val_loss += outputs.loss.item()
            
    avg_val_loss = total_val_loss / len(val_loader)
    print(f"Epoch {epoch} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
    
    ckpt_dir = f"/kaggle/working/sft_epoch_{epoch}"
    os.makedirs(ckpt_dir, exist_ok=True)
    model.save_pretrained(ckpt_dir)
    tokenizer.save_pretrained(ckpt_dir)

print("SFT Training Completed Successfully!")

# --- Cell 15: Read test questions ---
qa_df = pd.read_csv('Seed_Test_QA.csv')
print("Test questions loaded:", len(qa_df))

# --- Cell 16: Verify model checkpoints for evaluation ---
print("Checking available checkpoints in working directory:", os.listdir('/kaggle/working'))

# --- Cell 17: Load sft_epoch_3 and set evaluation seed ---
torch.manual_seed(42)
eval_model_path = '/kaggle/working/sft_epoch_3'
sft_tokenizer = GPT2Tokenizer.from_pretrained(eval_model_path)
sft_model = GPT2LMHeadModel.from_pretrained(eval_model_path)
sft_model.to(device)
print("SFT epoch 3 loaded for generation.")

# --- Cell 18: Generate SFT answers and save sft_results.csv ---
sft_model.eval()
results = []

for idx, row in qa_df.iterrows():
    q = row['Question']
    prompt = f"<|user|>{q}<|assistant|>"
    inputs = sft_tokenizer(prompt, return_tensors='pt').to(device)
    
    with torch.no_grad():
        output_ids = sft_model.generate(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask'],
            max_new_tokens=150,
            do_sample=False,
            pad_token_id=sft_tokenizer.pad_token_id,
            eos_token_id=sft_tokenizer.eos_token_id
        )
    
    generated_text = sft_tokenizer.decode(output_ids[0], skip_special_tokens=True)
    results.append({
        'Question': q,
        'Expected_Answer': row['Expected_Answer'],
        'Model_Answer': generated_text
    })

results_df = pd.DataFrame(results)
results_df.to_csv('/kaggle/working/sft_results.csv', index=False)
print("SFT Results saved successfully to sft_results.csv!")
display(results_df.head())